# Popolamento Neo4j + ChromaDB da TMDB CSV

Questo notebook carica i dati cinematografici da due CSV (TMDB 5000) e popola:
- **Neo4j** con nodi tipizzati (`Film`, `Director`, `Actor`, `Genre`) e le relative relazioni
- **ChromaDB** (via Agno) con embedding semantici delle trame, pronti per la ricerca vettoriale dell'agente

## 1 — Imports e Configurazione

In [2]:
import os
import json
import time
import logging

import pandas as pd
from dotenv import load_dotenv
from neo4j import GraphDatabase

from agno.vectordb.chroma import ChromaDb
from agno.knowledge import Knowledge
from agno.knowledge.document import Document
from agno.knowledge.embedder.google import GeminiEmbedder

# Silenzia i log INFO di agno (evita spam "Upserting 1 documents")
logging.getLogger("agno").setLevel(logging.WARNING)

load_dotenv()

# Radice del progetto: risale le cartelle a partire dalla working directory
# del kernel finche' non trova pyproject.toml. Necessario perche' il notebook
# vive in ingestion/ ma i percorsi (tmp/chromadb, .env) sono relativi alla radice:
# un percorso letterale "tmp/chromadb" si romperebbe se il kernel partisse con
# working directory = ingestion/ invece che la radice del progetto.
from pathlib import Path

def _find_project_root(marker: str = "pyproject.toml") -> Path:
    for parent in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (parent / marker).exists():
            return parent
    raise RuntimeError(f"Impossibile trovare la radice del progetto (marker: {marker})")

PROJECT_ROOT = _find_project_root()
print(f"Radice del progetto: {PROJECT_ROOT}")
print("Imports completati.")

Imports completati.


## 2 — Connessioni ai Database

In [2]:
# ── Neo4j ────────────────────────────────────────────────────────────────────
NEO4J_URI      = os.getenv("NEO4J_URI",      "bolt://localhost:7687")
NEO4J_USER     = os.getenv("NEO4J_USER",     "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD", "password")

driver_neo4j = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
try:
    driver_neo4j.verify_connectivity()
    print("Neo4j: connessione stabilita!")
except Exception as e:
    raise RuntimeError(f"Neo4j non raggiungibile: {e}")

# ── ChromaDB (via Agno) ──────────────────────────────────────────────────────
# task_type='RETRIEVAL_DOCUMENT' ottimizza gli embedding per l'indicizzazione.
# Quando l'agente esegue query, usa 'RETRIEVAL_QUERY' (default), che e'
# progettato per funzionare in coppia con 'RETRIEVAL_DOCUMENT'.
embedder = GeminiEmbedder(
    id="gemini-embedding-001",
    api_key=os.getenv("GEMINI_API_KEY"),
    task_type="RETRIEVAL_DOCUMENT",
    dimensions=1536,
)

vector_db = ChromaDb(
    collection="trame_cinema",
    path=str(PROJECT_ROOT / "tmp" / "chromadb"),
    persistent_client=True,
    embedder=embedder,
)
vector_db.create()  # crea la collection se non esiste, altrimenti la apre
print("ChromaDB: collection 'trame_cinema' pronta!")

# Questo e' l'oggetto Knowledge che l'agente usera' per le ricerche
knowledge_base = Knowledge(
    name="CinemaKB",
    vector_db=vector_db,
)
print("Knowledge base configurata.")

Neo4j: connessione stabilita!
ChromaDB: collection 'trame_cinema' pronta!
Knowledge base configurata.


## 3 — Caricamento e Pulizia dei Dati CSV

In [3]:
movies_df  = pd.read_csv("data/tmdb_5000_movies.csv")
credits_df = pd.read_csv("data/tmdb_5000_credits.csv")

# Unione su movie_id / id
df = movies_df.merge(credits_df, left_on="id", right_on="movie_id")
print(f"Film totali dopo il merge: {len(df)}")

# ── Funzioni di estrazione dai campi JSON annidati ────────────────────────────

def extract_top_actors(cast_str, top_n: int = 5) -> list[str]:
    """Restituisce i nomi dei top-N attori dal campo 'cast' (JSON stringa)."""
    try:
        return [a["name"] for a in json.loads(cast_str)[:top_n]]
    except Exception:
        return []

def extract_director(crew_str) -> str | None:
    """Restituisce il nome del regista (job == 'Director'), o None se assente."""
    try:
        for member in json.loads(crew_str):
            if member.get("job") == "Director":
                return member["name"]
    except Exception:
        pass
    return None

def extract_genres(genres_str) -> list[str]:
    """Restituisce la lista dei nomi di genere dal campo 'genres' (JSON stringa)."""
    try:
        return [g["name"] for g in json.loads(genres_str)]
    except Exception:
        return []

def safe_year(release_date) -> str | None:
    """Estrae l'anno da una stringa data 'YYYY-MM-DD', o None."""
    try:
        return str(release_date)[:4] if pd.notna(release_date) else None
    except Exception:
        return None

# ── Pulizia e arricchimento del DataFrame ─────────────────────────────────────
df["actors"]   = df["cast"].apply(extract_top_actors)
df["director"] = df["crew"].apply(extract_director)
df["genres"]   = df["genres"].apply(extract_genres)
df["year"]     = df["release_date"].apply(safe_year)

# Usa la colonna 'title' dal CSV movies (title_x dopo il merge)
df = df.rename(columns={"title_x": "title"})

# Rimuovi i film senza trama: non possono essere indicizzati in ChromaDB
df_clean = df[df["overview"].notna() & (df["overview"].str.strip() != "")].copy()
print(f"Film con trama valida: {len(df_clean)} / {len(df)}")

df_clean[["title", "director", "actors", "genres", "year"]].head(3)

Film totali dopo il merge: 4803
Film con trama valida: 4799 / 4803


,title,director,actors,genres,year
0,Avatar,James Cameron,"[Sam Worthington, Zoe Saldana, Sigourney Weave...","[Action, Adventure, Fantasy, Science Fiction]",2009
1,Pirates of the Caribbean: At World's End,Gore Verbinski,"[Johnny Depp, Orlando Bloom, Keira Knightley, ...","[Adventure, Fantasy, Action]",2007
2,Spectre,Sam Mendes,"[Daniel Craig, Christoph Waltz, Léa Seydoux, R...","[Action, Adventure, Crime]",2015


## 4 — Schema Neo4j: Constraint e Indici

I constraint garantiscono unicita' e velocizzano drasticamente i `MERGE`.

In [4]:
CONSTRAINTS = [
    "CREATE CONSTRAINT film_vector_id IF NOT EXISTS FOR (f:Film) REQUIRE f.vector_id IS UNIQUE",
    "CREATE CONSTRAINT director_nome    IF NOT EXISTS FOR (d:Director) REQUIRE d.nome IS UNIQUE",
    "CREATE CONSTRAINT actor_nome       IF NOT EXISTS FOR (a:Actor) REQUIRE a.nome IS UNIQUE",
    "CREATE CONSTRAINT genre_nome       IF NOT EXISTS FOR (g:Genre) REQUIRE g.nome IS UNIQUE",
]

with driver_neo4j.session() as session:
    for stmt in CONSTRAINTS:
        session.run(stmt)
        print(f"  OK  {stmt[:60]}...")

print("\nSchema Neo4j configurato.")

  OK  CREATE CONSTRAINT film_vector_id IF NOT EXISTS FOR (f:Film) ...
  OK  CREATE CONSTRAINT director_nome    IF NOT EXISTS FOR (d:Dire...
  OK  CREATE CONSTRAINT actor_nome       IF NOT EXISTS FOR (a:Acto...
  OK  CREATE CONSTRAINT genre_nome       IF NOT EXISTS FOR (g:Genr...

Schema Neo4j configurato.


## 5 — Funzioni di Ingestion

In [5]:
# ── Neo4j ─────────────────────────────────────────────────────────────────────

CYPHER_MERGE_FILM = """
MERGE (f:Film {vector_id: $vector_id})
SET   f.titolo = $titolo,
      f.anno   = $anno,
      f.trama  = $trama

WITH f

FOREACH (nome IN CASE WHEN $regista IS NOT NULL THEN [$regista] ELSE [] END |
    MERGE (d:Director {nome: nome})
    MERGE (d)-[:DIRECTED]->(f)
)

WITH f
UNWIND $generi AS genere_nome
    MERGE (g:Genre {nome: genere_nome})
    MERGE (f)-[:HAS_GENRE]->(g)

WITH DISTINCT f
UNWIND $attori AS attore_nome
    MERGE (a:Actor {nome: attore_nome})
    MERGE (a)-[:ACTED_IN]->(f)
"""

def salva_film_neo4j(session, row: dict) -> None:
    """Inserisce (o aggiorna) un film e le sue relazioni nel grafo Neo4j."""
    session.run(
        CYPHER_MERGE_FILM,
        vector_id = str(row["id"]),
        titolo    = row["title"]    or "",
        anno      = row["year"],
        trama     = (row["overview"] or "")[:2000],  # limita lunghezza proprieta'
        regista   = row["director"],  # None se assente — gestito dal FOREACH
        generi    = row["genres"]  or [],
        attori    = row["actors"]  or [],
    )


# ── ChromaDB ──────────────────────────────────────────────────────────────────

def crea_documento_film(row: dict) -> Document:
    """
    Costruisce un Document Agno da una riga del DataFrame.
    Il testo e' arricchito con titolo, regista, cast e generi per fornire
    contesto semantico all'embedding — non solo la trama grezza.
    """
    actors_str = ", ".join(row["actors"])  if row["actors"]  else "N/A"
    genres_str = ", ".join(row["genres"])  if row["genres"]  else "N/A"
    director   = row["director"] or "Sconosciuto"

    testo = (
        f"Titolo: {row['title']}\n"
        f"Anno: {row['year']}\n"
        f"Regista: {director}\n"
        f"Attori principali: {actors_str}\n"
        f"Generi: {genres_str}\n"
        f"Trama: {row['overview']}"
    )

    return Document(
        id=str(row["id"]),
        name=row["title"],
        content=testo,
        meta_data={
            "movie_id": str(row["id"]),
            "title":    row["title"] or "",
            "director": director,
            "year":     row["year"]  or "",
            "genres":   genres_str,
            "actors":   actors_str,
        },
    )

print("Funzioni di ingestion definite.")

Funzioni di ingestion definite.


## 6 — Popolamento Neo4j

In [ ]:
records = df_clean.to_dict(orient="records")
LIMIT = None  # imposta un intero (es. 100) per test rapidi, None per tutto
if LIMIT:
    records = records[:LIMIT]

totale = len(records)
neo4j_ok = 0
neo4j_err = 0

with driver_neo4j.session() as neo4j_session:
    for i, row in enumerate(records):
        try:
            salva_film_neo4j(neo4j_session, row)
            neo4j_ok += 1
        except Exception as exc:
            neo4j_err += 1
            print(f"[Neo4j] Errore '{row.get('title')}': {exc}")

        if (i + 1) % 200 == 0 or (i + 1) == totale:
            print(f"  {i + 1}/{totale} — ok:{neo4j_ok}  errori:{neo4j_err}")

print(f"\nNeo4j completato — inseriti: {neo4j_ok}  errori: {neo4j_err}")

## 7 — Preparazione Documenti per ChromaDB (nessuna chiamata API)

In [6]:
# Definisce records se non già presente (es. se la cella 6 è stata saltata)
if "records" not in dir():
    records = df_clean.to_dict(orient="records")

# Crea tutti i Document senza ancora chiamare l'API Gemini.
# L'embedding avviene nella cella successiva, dentro ChromaDB.upsert().
documents = []
doc_err = 0

for row in records:
    try:
        documents.append(crea_documento_film(row))
    except Exception as exc:
        doc_err += 1
        print(f"[Doc] Errore '{row.get('title')}': {exc}")

print(f"Documenti preparati: {len(documents)}  errori: {doc_err}")

Documenti preparati: 4799  errori: 0


## 8 — Embedding + Inserimento ChromaDB (chiama API Gemini)

In [7]:
MAX_RETRIES       = 3
BASE_RETRY_SLEEP  = 35  # secondi di attesa al primo 429 (raddoppia a ogni retry)

chroma_ok   = 0
chroma_skip = 0
chroma_err  = 0
stop        = False  # flag per uscita pulita su quota esaurita

for i, doc in enumerate(documents):
    if stop:
        break

    content_hash = f"movie_{doc.id}"

    if vector_db.content_hash_exists(content_hash):
        chroma_skip += 1
        continue

    for attempt in range(MAX_RETRIES):
        try:
            vector_db.upsert(content_hash=content_hash, documents=[doc])
            chroma_ok += 1
            break
        except Exception as exc:
            if "429" in str(exc):
                if attempt < MAX_RETRIES - 1:
                    wait = BASE_RETRY_SLEEP * (2 ** attempt)
                    print(f"  [429] Attendo {wait}s prima di riprovare...")
                    time.sleep(wait)
                else:
                    print(f"  [STOP] Quota giornaliera esaurita dopo {chroma_ok} embedding.")
                    print(f"         Riesegui domani: i {chroma_ok} film inseriti verranno saltati.")
                    chroma_err += len(documents) - i
                    stop = True
                    break
            else:
                chroma_err += 1
                print(f"  [Errore] {doc.name}: {exc}")
                break

    done = chroma_ok + chroma_skip + chroma_err
    if done % 100 == 0 or done == len(documents):
        print(f"  {done}/{len(documents)} — inseriti:{chroma_ok}  saltati:{chroma_skip}  errori:{chroma_err}")

print(f"\nChromaDB completato — inseriti:{chroma_ok}  saltati:{chroma_skip}  errori:{chroma_err}")
print(f"Totale documenti in collection: {vector_db.get_count()}")

  4000/4799 — inseriti:5  saltati:3995  errori:0


ERROR    Error upserting documents by content hash                                                                 
         Traceback (most recent call last):                                                                        
           File "/Users/piermarone/Desktop/Progetto                                                                
         AgenticAI/.venv/lib/python3.13/site-packages/agno/vectordb/chroma/chromadb.py", line 495, in upsert       
             self._upsert(content_hash, documents, filters)                                                        
             ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^                                                        
           File "/Users/piermarone/Desktop/Progetto                                                                
         AgenticAI/.venv/lib/python3.13/site-packages/agno/vectordb/chroma/chromadb.py", line 518, in _upsert      
             document.embed(embedder=self.embedder)                                                                
             ~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^                                                                
           File "/Users/piermarone/Desktop/Progetto                                                                
         AgenticAI/.venv/lib/python3.13/site-packages/agno/knowledge/document/base.py", line 30, in embed          
             self.embedding, self.usage = _embedder.get_embedding_and_usage(self.content)                          
                                          ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^                          
           File "/Users/piermarone/Desktop/Progetto                                                                
         AgenticAI/.venv/lib/python3.13/site-packages/agno/knowledge/embedder/google.py", line 101, in             
         get_embedding_and_usage                                                                                   
             response = self._response(text=text)                                                                  
           File "/Users/piermarone/Desktop/Progetto                                                                
         AgenticAI/.venv/lib/python3.13/site-packages/agno/knowledge/embedder/google.py", line 85, in _response    
             return self.client.models.embed_content(**_request_params)                                            
                    ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^                                            
           File "/Users/piermarone/Desktop/Progetto                                                                
         AgenticAI/.venv/lib/python3.13/site-packages/google/genai/models.py", line 6265, in embed_content         
             return self._embed_content(model=model, contents=contents, config=config)                             
                    ~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^                             
           File "/Users/piermarone/Desktop/Progetto                                                                
         AgenticAI/.venv/lib/python3.13/site-packages/google/genai/models.py", line 5070, in _embed_content        
             response = self._api_client.request(                                                                  
                 'post', path, request_dict, http_options                                                          
             )                                                                                                     
           File "/Users/piermarone/Desktop/Progetto                                                                
         AgenticAI/.venv/lib/python3.13/site-packages/google/genai/_api_client.py", line 1611, in request          
             response = self._request(http_request, http_options, stream=False)                                    
           File "/Users/piermarone/Desktop/Progetto     

  [429] Attendo 35s prima di riprovare...
  4100/4799 — inseriti:105  saltati:3995  errori:0


ERROR    Error upserting documents by content hash                                                                 
         Traceback (most recent call last):                                                                        
           File "/Users/piermarone/Desktop/Progetto                                                                
         AgenticAI/.venv/lib/python3.13/site-packages/agno/vectordb/chroma/chromadb.py", line 495, in upsert       
             self._upsert(content_hash, documents, filters)                                                        
             ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^                                                        
           File "/Users/piermarone/Desktop/Progetto                                                                
         AgenticAI/.venv/lib/python3.13/site-packages/agno/vectordb/chroma/chromadb.py", line 518, in _upsert      
             document.embed(embedder=self.embedder)                                                                
             ~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^                                                                
           File "/Users/piermarone/Desktop/Progetto                                                                
         AgenticAI/.venv/lib/python3.13/site-packages/agno/knowledge/document/base.py", line 30, in embed          
             self.embedding, self.usage = _embedder.get_embedding_and_usage(self.content)                          
                                          ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^                          
           File "/Users/piermarone/Desktop/Progetto                                                                
         AgenticAI/.venv/lib/python3.13/site-packages/agno/knowledge/embedder/google.py", line 101, in             
         get_embedding_and_usage                                                                                   
             response = self._response(text=text)                                                                  
           File "/Users/piermarone/Desktop/Progetto                                                                
         AgenticAI/.venv/lib/python3.13/site-packages/agno/knowledge/embedder/google.py", line 85, in _response    
             return self.client.models.embed_content(**_request_params)                                            
                    ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^                                            
           File "/Users/piermarone/Desktop/Progetto                                                                
         AgenticAI/.venv/lib/python3.13/site-packages/google/genai/models.py", line 6265, in embed_content         
             return self._embed_content(model=model, contents=contents, config=config)                             
                    ~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^                             
           File "/Users/piermarone/Desktop/Progetto                                                                
         AgenticAI/.venv/lib/python3.13/site-packages/google/genai/models.py", line 5070, in _embed_content        
             response = self._api_client.request(                                                                  
                 'post', path, request_dict, http_options                                                          
             )                                                                                                     
           File "/Users/piermarone/Desktop/Progetto                                                                
         AgenticAI/.venv/lib/python3.13/site-packages/google/genai/_api_client.py", line 1611, in request          
             response = self._request(http_request, http_options, stream=False)                                    
           File "/Users/piermarone/Desktop/Progetto     

  [429] Attendo 35s prima di riprovare...
  4200/4799 — inseriti:205  saltati:3995  errori:0
  4300/4799 — inseriti:305  saltati:3995  errori:0


...

  [429] Attendo 35s prima di riprovare...
  4500/4799 — inseriti:505  saltati:3995  errori:0


ERROR    Error upserting documents by content hash                                                                 
         Traceback (most recent call last):                                                                        
           File "/Users/piermarone/Desktop/Progetto                                                                
         AgenticAI/.venv/lib/python3.13/site-packages/agno/vectordb/chroma/chromadb.py", line 495, in upsert       
             self._upsert(content_hash, documents, filters)                                                        
             ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^                                                        
           File "/Users/piermarone/Desktop/Progetto                                                                
         AgenticAI/.venv/lib/python3.13/site-packages/agno/vectordb/chroma/chromadb.py", line 518, in _upsert      
             document.embed(embedder=self.embedder)                                                                
             ~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^                                                                
           File "/Users/piermarone/Desktop/Progetto                                                                
         AgenticAI/.venv/lib/python3.13/site-packages/agno/knowledge/document/base.py", line 30, in embed          
             self.embedding, self.usage = _embedder.get_embedding_and_usage(self.content)                          
                                          ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^                          
           File "/Users/piermarone/Desktop/Progetto                                                                
         AgenticAI/.venv/lib/python3.13/site-packages/agno/knowledge/embedder/google.py", line 101, in             
         get_embedding_and_usage                                                                                   
             response = self._response(text=text)                                                                  
           File "/Users/piermarone/Desktop/Progetto                                                                
         AgenticAI/.venv/lib/python3.13/site-packages/agno/knowledge/embedder/google.py", line 85, in _response    
             return self.client.models.embed_content(**_request_params)                                            
                    ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^                                            
           File "/Users/piermarone/Desktop/Progetto                                                                
         AgenticAI/.venv/lib/python3.13/site-packages/google/genai/models.py", line 6265, in embed_content         
             return self._embed_content(model=model, contents=contents, config=config)                             
                    ~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^                             
           File "/Users/piermarone/Desktop/Progetto                                                                
         AgenticAI/.venv/lib/python3.13/site-packages/google/genai/models.py", line 5070, in _embed_content        
             response = self._api_client.request(                                                                  
                 'post', path, request_dict, http_options                                                          
             )                                                                                                     
           File "/Users/piermarone/Desktop/Progetto                                                                
         AgenticAI/.venv/lib/python3.13/site-packages/google/genai/_api_client.py", line 1611, in request          
             response = self._request(http_request, http_options, stream=False)                                    
           File "/Users/piermarone/Desktop/Progetto     

  [429] Attendo 35s prima di riprovare...
  4600/4799 — inseriti:605  saltati:3995  errori:0
  4700/4799 — inseriti:705  saltati:3995  errori:0


ERROR    Error upserting documents by content hash                                                                 
         Traceback (most recent call last):                                                                        
           File "/Users/piermarone/Desktop/Progetto                                                                
         AgenticAI/.venv/lib/python3.13/site-packages/agno/vectordb/chroma/chromadb.py", line 495, in upsert       
             self._upsert(content_hash, documents, filters)                                                        
             ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^                                                        
           File "/Users/piermarone/Desktop/Progetto                                                                
         AgenticAI/.venv/lib/python3.13/site-packages/agno/vectordb/chroma/chromadb.py", line 518, in _upsert      
             document.embed(embedder=self.embedder)                                                                
             ~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^                                                                
           File "/Users/piermarone/Desktop/Progetto                                                                
         AgenticAI/.venv/lib/python3.13/site-packages/agno/knowledge/document/base.py", line 30, in embed          
             self.embedding, self.usage = _embedder.get_embedding_and_usage(self.content)                          
                                          ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^                          
           File "/Users/piermarone/Desktop/Progetto                                                                
         AgenticAI/.venv/lib/python3.13/site-packages/agno/knowledge/embedder/google.py", line 101, in             
         get_embedding_and_usage                                                                                   
             response = self._response(text=text)                                                                  
           File "/Users/piermarone/Desktop/Progetto                                                                
         AgenticAI/.venv/lib/python3.13/site-packages/agno/knowledge/embedder/google.py", line 85, in _response    
             return self.client.models.embed_content(**_request_params)                                            
                    ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^                                            
           File "/Users/piermarone/Desktop/Progetto                                                                
         AgenticAI/.venv/lib/python3.13/site-packages/google/genai/models.py", line 6265, in embed_content         
             return self._embed_content(model=model, contents=contents, config=config)                             
                    ~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^                             
           File "/Users/piermarone/Desktop/Progetto                                                                
         AgenticAI/.venv/lib/python3.13/site-packages/google/genai/models.py", line 5070, in _embed_content        
             response = self._api_client.request(                                                                  
                 'post', path, request_dict, http_options                                                          
             )                                                                                                     
           File "/Users/piermarone/Desktop/Progetto                                                                
         AgenticAI/.venv/lib/python3.13/site-packages/google/genai/_api_client.py", line 1611, in request          
             response = self._request(http_request, http_options, stream=False)                                    
           File "/Users/piermarone/Desktop/Progetto     

  [429] Attendo 35s prima di riprovare...
  4799/4799 — inseriti:804  saltati:3995  errori:0

ChromaDB completato — inseriti:804  saltati:3995  errori:0
Totale documenti in collection: 4801


## 9 — Verifica e Statistiche

In [ ]:
# ── Verifica Neo4j ────────────────────────────────────────────────────────────
COUNT_QUERY = """
MATCH (f:Film)     WITH count(f) AS films
MATCH (d:Director) WITH films, count(d) AS directors
MATCH (a:Actor)    WITH films, directors, count(a) AS actors
MATCH (g:Genre)    RETURN films, directors, actors, count(g) AS genres
"""

with driver_neo4j.session() as session:
    result = session.run(COUNT_QUERY).single()

print("Neo4j — nodi presenti nel grafo:")
print(f"  Film:     {result['films']}")
print(f"  Director: {result['directors']}")
print(f"  Actor:    {result['actors']}")
print(f"  Genre:    {result['genres']}")

# Esempio: film piu' costosi con il loro regista
SAMPLE_QUERY = """
MATCH (d:Director)-[:DIRECTED]->(f:Film)
RETURN f.titolo AS titolo, f.anno AS anno, d.nome AS regista
ORDER BY f.titolo
LIMIT 5
"""
print("\nCampione di film nel grafo:")
with driver_neo4j.session() as session:
    for record in session.run(SAMPLE_QUERY):
        print(f"  {record['titolo']} ({record['anno']}) — regia: {record['regista']}")

In [ ]:
# ── Verifica ChromaDB ─────────────────────────────────────────────────────────
chroma_count = vector_db.get_count()
print(f"ChromaDB — documenti nella collection 'trame_cinema': {chroma_count}")

# Test di ricerca semantica
print("\nTest ricerca semantica — query: 'astronauts lost in space'")
risultati = vector_db.search(query="astronauts lost in space", limit=3)
for i, doc in enumerate(risultati, 1):
    meta = doc.meta_data
    print(f"  {i}. {meta.get('title')} ({meta.get('year')}) — {meta.get('director')}")

In [ ]:
# ── Verifica relazionale Neo4j: attori che hanno lavorato con lo stesso regista ─
COLLAB_QUERY = """
MATCH (d:Director)-[:DIRECTED]->(f:Film)<-[:ACTED_IN]-(a:Actor)
WITH d.nome AS regista, collect(DISTINCT a.nome) AS cast_totale, count(DISTINCT f) AS num_film
WHERE num_film >= 3
RETURN regista, num_film, cast_totale[..5] AS top_attori
ORDER BY num_film DESC
LIMIT 5
"""

print("Registi con piu' film nel grafo (minimo 3):")
with driver_neo4j.session() as session:
    for rec in session.run(COLLAB_QUERY):
        print(f"  {rec['regista']} ({rec['num_film']} film) — attori: {rec['top_attori']}")

## 10 — Biografie da TMDB API

Prerequisito: aggiungi `TMDB_API_KEY=...` al file `.env`  
Registrazione gratuita su https://www.themoviedb.org/settings/api

Le biografie vengono:
- Scritte come proprieta' sui nodi `Actor` e `Director` in Neo4j
- Indicizzate come documenti separati in ChromaDB per la ricerca semantica

Il checkpoint e' basato sulla proprieta' `tmdb_id` del nodo Neo4j:
se gia' presente, la persona viene saltata nelle esecuzioni successive.

In [3]:
import requests

# Ricarica .env per includere TMDB_API_KEY aggiunta dopo l'avvio del kernel
load_dotenv(dotenv_path=str(PROJECT_ROOT / ".env"), override=True)

TMDB_API_KEY = os.getenv("TMDB_API_KEY")
if not TMDB_API_KEY:
    raise ValueError("TMDB_API_KEY mancante nel file .env")

TMDB_BASE = "https://api.themoviedb.org/3"
TMDB_HEADERS = {"Authorization": f"Bearer {TMDB_API_KEY}", "accept": "application/json"}
TMDB_SLEEP = 0.05  # 20 req/s (limite ufficiale: 40/s)

# ── Estrai tutte le persone uniche dal CSV con i loro TMDB person_id ──────────
persons = {}  # {tmdb_person_id: {"name": ..., "role": ...}}

credits_raw = pd.read_csv("data/tmdb_5000_credits.csv")

for _, row in credits_raw.iterrows():
    try:
        for a in json.loads(row["cast"])[:5]:
            pid = a["id"]
            if pid not in persons:
                persons[pid] = {"name": a["name"], "role": "actor"}
    except Exception:
        pass
    try:
        for c in json.loads(row["crew"]):
            if c.get("job") == "Director":
                pid = c["id"]
                if pid not in persons:
                    persons[pid] = {"name": c["name"], "role": "director"}
    except Exception:
        pass

print(f"Persone uniche trovate nel CSV: {len(persons)}")
print(f"  Attori:   {sum(1 for p in persons.values() if p['role'] == 'actor')}")
print(f"  Registi:  {sum(1 for p in persons.values() if p['role'] == 'director')}")

# Test rapido connessione TMDB
test = requests.get(f"{TMDB_BASE}/person/65731", headers=TMDB_HEADERS, timeout=10)
print(f"\nTest TMDB API: {test.status_code} — {test.json().get('name')}")

Persone uniche trovate nel CSV: 11718
  Attori:   9337
  Registi:  2381

Test TMDB API: 200 — Sam Worthington


### 10a — Fetch TMDB → aggiornamento Neo4j

Checkpoint: le persone con `tmdb_id` gia' impostato in Neo4j vengono saltate.  
Riesegui liberamente questa cella: riprende sempre da dove si era fermata.

In [10]:
CYPHER_UPDATE_ACTOR = """
MATCH (a:Actor {nome: $nome})
SET a.tmdb_id       = $tmdb_id,
    a.biografia     = $biografia,
    a.data_nascita  = $data_nascita,
    a.luogo_nascita = $luogo_nascita
"""

CYPHER_UPDATE_DIRECTOR = """
MATCH (d:Director {nome: $nome})
SET d.tmdb_id       = $tmdb_id,
    d.biografia     = $biografia,
    d.data_nascita  = $data_nascita,
    d.luogo_nascita = $luogo_nascita
"""

def fetch_tmdb_person(person_id: int) -> dict | None:
    """Chiama TMDB /person/{id} e restituisce i campi utili, o None su errore."""
    url = f"{TMDB_BASE}/person/{person_id}"
    try:
        resp = requests.get(url, headers=TMDB_HEADERS, timeout=10)
        if resp.status_code == 404:
            return None
        resp.raise_for_status()
        data = resp.json()
        return {
            "biografia":     (data.get("biography") or "").strip(),
            "data_nascita":  data.get("birthday") or "",
            "luogo_nascita": data.get("place_of_birth") or "",
        }
    except Exception as exc:
        print(f"  [TMDB] Errore person {person_id}: {exc}")
        return None


# ── Leggi i tmdb_id gia' presenti in Neo4j (checkpoint) ──────────────────────
with driver_neo4j.session() as s:
    already_actors    = {r["nome"] for r in s.run("MATCH (a:Actor)    WHERE a.tmdb_id IS NOT NULL RETURN a.nome AS nome")}
    already_directors = {r["nome"] for r in s.run("MATCH (d:Director) WHERE d.tmdb_id IS NOT NULL RETURN d.nome AS nome")}

print(f"Gia' aggiornati — Attori: {len(already_actors)}  Registi: {len(already_directors)}")

# ── Loop principale ───────────────────────────────────────────────────────────
ok = 0
skip = 0
err = 0
bio_docs = []  # documenti da inserire in ChromaDB (cella successiva)

totale = len(persons)

with driver_neo4j.session() as neo4j_session:
    for i, (pid, info) in enumerate(persons.items()):
        nome = info["name"]
        role = info["role"]

        # Checkpoint
        if role == "actor" and nome in already_actors:
            skip += 1
        elif role == "director" and nome in already_directors:
            skip += 1
        else:
            data = fetch_tmdb_person(pid)
            time.sleep(TMDB_SLEEP)

            if data is None:
                err += 1
            else:
                cypher = CYPHER_UPDATE_ACTOR if role == "actor" else CYPHER_UPDATE_DIRECTOR
                neo4j_session.run(cypher, nome=nome, tmdb_id=pid, **data)
                ok += 1

                # Prepara documento per ChromaDB solo se c'e' una biografia
                if data["biografia"]:
                    testo = (
                        f"Nome: {nome}\n"
                        f"Ruolo: {'Attore' if role == 'actor' else 'Regista'}\n"
                        f"Nato il: {data['data_nascita']} a {data['luogo_nascita']}\n"
                        f"Biografia: {data['biografia']}"
                    )
                    bio_docs.append(Document(
                        id=f"bio_{pid}",
                        name=nome,
                        content=testo,
                        meta_data={
                            "type":      "persona",
                            "person_id": str(pid),
                            "name":      nome,
                            "role":      role,
                        },
                    ))

        if (i + 1) % 500 == 0 or (i + 1) == totale:
            print(f"  {i + 1}/{totale} — ok:{ok}  saltati:{skip}  errori:{err}  bio_pronte:{len(bio_docs)}")

print(f"\nNeo4j biografie completate — ok:{ok}  saltati:{skip}  errori:{err}")
print(f"Documenti biografia pronti per ChromaDB: {len(bio_docs)}")

Gia' aggiornati — Attori: 9237  Registi: 2143
  500/11718 — ok:9  saltati:491  errori:0  bio_pronte:9
  1000/11718 — ok:18  saltati:982  errori:0  bio_pronte:14
  1500/11718 — ok:30  saltati:1470  errori:0  bio_pronte:23
  2000/11718 — ok:40  saltati:1960  errori:0  bio_pronte:31
  2500/11718 — ok:48  saltati:2452  errori:0  bio_pronte:34
  3000/11718 — ok:55  saltati:2943  errori:2  bio_pronte:40
  3500/11718 — ok:63  saltati:3434  errori:3  bio_pronte:47
  4000/11718 — ok:70  saltati:3924  errori:6  bio_pronte:51
  4500/11718 — ok:80  saltati:4411  errori:9  bio_pronte:54
  5000/11718 — ok:86  saltati:4901  errori:13  bio_pronte:58
  5500/11718 — ok:94  saltati:5391  errori:15  bio_pronte:61
  6000/11718 — ok:109  saltati:5873  errori:18  bio_pronte:71
  6500/11718 — ok:119  saltati:6358  errori:23  bio_pronte:77
  7000/11718 — ok:129  saltati:6841  errori:30  bio_pronte:83
  7500/11718 — ok:140  saltati:7326  errori:34  bio_pronte:92
  8000/11718 — ok:147  saltati:7815  errori:38  b

### 10a-bis — Ricostruzione `bio_docs` da Neo4j (senza richiamare TMDB)

Da usare **solo** se il kernel e' stato riavviato dopo l'esecuzione della 10a: `bio_docs` e' andato perso, ma le biografie sono gia' salvate su Neo4j.  
Questa cella le rilegge direttamente dai nodi `Actor`/`Director` (nessuna chiamata a TMDB) e ricostruisce `bio_docs` per la 10b.  
Se il kernel e' rimasto attivo dalla 10a e `bio_docs` e' gia' popolato, salta questa cella.

In [ ]:
with driver_neo4j.session() as s:
    rows = s.run("""
        MATCH (a:Actor)
        WHERE a.biografia IS NOT NULL AND a.biografia <> ''
        RETURN a.nome AS nome, a.tmdb_id AS tmdb_id, a.biografia AS biografia,
               a.data_nascita AS data_nascita, a.luogo_nascita AS luogo_nascita,
               'actor' AS role
        UNION
        MATCH (d:Director)
        WHERE d.biografia IS NOT NULL AND d.biografia <> ''
        RETURN d.nome AS nome, d.tmdb_id AS tmdb_id, d.biografia AS biografia,
               d.data_nascita AS data_nascita, d.luogo_nascita AS luogo_nascita,
               'director' AS role
    """)
    bio_rows = [dict(r) for r in rows]

bio_docs = []
for r in bio_rows:
    testo = (
        f"Nome: {r['nome']}\n"
        f"Ruolo: {'Attore' if r['role'] == 'actor' else 'Regista'}\n"
        f"Nato il: {r['data_nascita'] or ''} a {r['luogo_nascita'] or ''}\n"
        f"Biografia: {r['biografia']}"
    )
    bio_docs.append(Document(
        id=f"bio_{r['tmdb_id']}",
        name=r['nome'],
        content=testo,
        meta_data={
            "type":      "persona",
            "person_id": str(r['tmdb_id']),
            "name":      r['nome'],
            "role":      r['role'],
        },
    ))

print(f"bio_docs ricostruiti da Neo4j: {len(bio_docs)}")

### 10b — Embedding biografie in ChromaDB

Stessa logica della cella 8: checkpoint per-documento, retry su 429.  
Soggetta al limite giornaliero Gemini (1000/giorno). Riesegui nei giorni successivi.

In [ ]:
import logging
logging.getLogger("agno").setLevel(logging.WARNING)

MAX_RETRIES      = 3
BASE_RETRY_SLEEP = 35  # secondi di attesa al primo 429 (raddoppia a ogni retry)

bio_ok   = 0
bio_skip = 0
bio_err  = 0
stop     = False

for i, doc in enumerate(bio_docs):
    if stop:
        break

    content_hash = f"bio_{doc.id.replace('bio_', '')}"

    if vector_db.content_hash_exists(content_hash):
        bio_skip += 1
        continue

    for attempt in range(MAX_RETRIES):
        try:
            vector_db.upsert(content_hash=content_hash, documents=[doc])
            bio_ok += 1
            break
        except Exception as exc:
            if "429" in str(exc):
                if attempt < MAX_RETRIES - 1:
                    wait = BASE_RETRY_SLEEP * (2 ** attempt)
                    print(f"  [429] Attendo {wait}s prima di riprovare...")
                    time.sleep(wait)
                else:
                    print(f"  [STOP] Quota esaurita dopo {bio_ok} biografie.")
                    print(f"         Riesegui domani: le {bio_ok} inserite verranno saltate.")
                    bio_err += len(bio_docs) - i
                    stop = True
                    break
            else:
                bio_err += 1
                print(f"  [Errore] {doc.name}: {exc}")
                break

    done = bio_ok + bio_skip + bio_err
    if done % 200 == 0 or done == len(bio_docs):
        print(f"  {done}/{len(bio_docs)} — inseriti:{bio_ok}  saltati:{bio_skip}  errori:{bio_err}")

print(f"\nBiografie ChromaDB — inserite:{bio_ok}  saltate:{bio_skip}  errori:{bio_err}")
print(f"Totale documenti in collection: {vector_db.get_count()}")

## 11 — Export biografie in CSV

In [ ]:
with driver_neo4j.session() as s:
    attori = s.run("""
        MATCH (a:Actor) WHERE a.biografia IS NOT NULL AND a.biografia <> ''
        RETURN 'attore' AS ruolo, a.nome AS nome,
               a.data_nascita AS data_nascita, a.luogo_nascita AS luogo_nascita,
               a.biografia AS biografia
        ORDER BY a.nome
    """).data()

    registi = s.run("""
        MATCH (d:Director) WHERE d.biografia IS NOT NULL AND d.biografia <> ''
        RETURN 'regista' AS ruolo, d.nome AS nome,
               d.data_nascita AS data_nascita, d.luogo_nascita AS luogo_nascita,
               d.biografia AS biografia
        ORDER BY d.nome
    """).data()

bio_df = pd.DataFrame(attori + registi)
bio_df.to_csv("data/biografie.csv", index=False, encoding="utf-8")

print(f"Esportati: {len(attori)} attori + {len(registi)} registi = {len(bio_df)} righe")
print(f"File: data/biografie.csv")
bio_df[["ruolo", "nome", "data_nascita", "luogo_nascita"]].head(10)

## 12 — Riepilogo: cosa rieseguire nelle prossime sessioni

Stato al 26/08/2026:
- **Film (sezione 8): completata** — 4799/4799 embeddati in ChromaDB, Neo4j popolato.
- **Biografie (10b): in corso** — riprende da dove si e' fermata grazie al checkpoint (`content_hash_exists`), limitata dalla quota giornaliera gratuita Gemini (~1000 embedding/giorno).

### Se il kernel viene riavviato (es. sessione del giorno dopo), eseguire in ordine SOLO queste celle:

1. **Sezione 1 — Imports** (cella con `import os, json, time, ...`)
2. **Sezione 2 — Connessioni** (Neo4j + ChromaDB/`vector_db` + `knowledge_base`)
3. **10a-bis — Ricostruzione `bio_docs` da Neo4j** (rilegge le biografie gia' salvate, nessuna chiamata TMDB)
4. **10b — Embedding biografie in ChromaDB** (riprende automaticamente da dove si era fermata)

### NON serve rieseguire (dati gia' persistiti):

- Sezione 3 (caricamento/pulizia CSV)
- Sezione 4 (schema/constraint Neo4j — idempotente comunque, ma inutile)
- Sezione 5 (funzioni di ingestion)
- Sezione 6 (popolamento Neo4j film)
- Sezione 7 (preparazione documenti film)
- **Sezione 8 (embedding film) — completata al 100%, da NON rieseguire**
- Sezione 9 (verifica/statistiche — opzionale, solo per controllo)
- Sezione 10 e 10a (fetch TMDB persone) — gia' fatto, sostituito dalla 10a-bis

### Quando la 10b avra' finito tutte le biografie:

- Eseguire la Sezione 11 (Export biografie in CSV) se serve un backup fuori da Neo4j/ChromaDB.